# 02. Run DROID-W Baseline vs Ours

Generated clean workflow notebook for `ZexuZang/vggt-demo`.


## Purpose
Run two SLAM experiments without overwriting results:
1. DROID-W + Ours static weights
2. Original DROID-W baseline

Outputs are copied to:
- `/content/vggt-demo/outputs_compare/droidw_ours`
- `/content/vggt-demo/outputs_compare/droidw_baseline`


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
%cd /content/vggt-demo
!git pull origin main


In [ ]:
from pathlib import Path

DROIDW_ROOT = Path('/content/vggt-demo/third_party/DROID-W')
OURS_WEIGHT_DIR = Path('/content/vggt-demo/outputs/droidw_ours_weights/person_tracking/static_weight_npy')

print('DROID-W root:', DROIDW_ROOT, DROIDW_ROOT.exists())
print('Ours static weight dir:', OURS_WEIGHT_DIR, OURS_WEIGHT_DIR.exists())
if OURS_WEIGHT_DIR.exists():
    files = sorted(OURS_WEIGHT_DIR.glob('*.npy'))
    print('Number of static weight files:', len(files))
    print('First files:', files[:5])
else:
    raise FileNotFoundError('Static weights not found. Run 01_generate_ours_static_weight.ipynb first.')


In [ ]:
%cd /content/vggt-demo/third_party/DROID-W

print('==== Check configs ====')
!ls configs/Dynamic/Bonn

print('
==== Baseline config should NOT contain ours_dynamic/static_weight ====')
!grep -n "ours_dynamic\|static_weight" configs/Dynamic/Bonn/bonn_person_tracking.yaml || true

print('
==== Ours config should contain ours_dynamic/static_weight ====')
!grep -n "ours_dynamic\|static_weight" configs/Dynamic/Bonn/person_tracking_ours.yaml || true


In [ ]:
%%bash
set -e
cd /content/vggt-demo/third_party/DROID-W
TORCH_LIB=$(python - <<'PY'
import torch, os
print(os.path.join(os.path.dirname(torch.__file__), 'lib'))
PY
)
export LD_LIBRARY_PATH="$TORCH_LIB:$LD_LIBRARY_PATH"
export PYTHONPATH="/content/vggt-demo/third_party/DROID-W:$PYTHONPATH"
rm -rf Outputs/Bonn/bonn_person_tracking
echo "==== Run DROID-W + Ours static weight ===="
python run.py --config configs/Dynamic/Bonn/person_tracking_ours.yaml
mkdir -p /content/vggt-demo/outputs_compare
rm -rf /content/vggt-demo/outputs_compare/droidw_ours
cp -a Outputs/Bonn/bonn_person_tracking /content/vggt-demo/outputs_compare/droidw_ours
SAVE_ROOT="/content/drive/MyDrive/vggt_demo_results/bonn_person_tracking_compare"
mkdir -p "$SAVE_ROOT"
rm -rf "$SAVE_ROOT/droidw_ours"
cp -a Outputs/Bonn/bonn_person_tracking "$SAVE_ROOT/droidw_ours"
echo "Saved Ours to /content/vggt-demo/outputs_compare/droidw_ours"
echo "Saved Ours backup to $SAVE_ROOT/droidw_ours"


In [ ]:
%%bash
set -e
cd /content/vggt-demo/third_party/DROID-W
TORCH_LIB=$(python - <<'PY'
import torch, os
print(os.path.join(os.path.dirname(torch.__file__), 'lib'))
PY
)
export LD_LIBRARY_PATH="$TORCH_LIB:$LD_LIBRARY_PATH"
export PYTHONPATH="/content/vggt-demo/third_party/DROID-W:$PYTHONPATH"
rm -rf Outputs/Bonn/bonn_person_tracking
echo "==== Run original DROID-W baseline ===="
python run.py --config configs/Dynamic/Bonn/bonn_person_tracking.yaml
mkdir -p /content/vggt-demo/outputs_compare
rm -rf /content/vggt-demo/outputs_compare/droidw_baseline
cp -a Outputs/Bonn/bonn_person_tracking /content/vggt-demo/outputs_compare/droidw_baseline
SAVE_ROOT="/content/drive/MyDrive/vggt_demo_results/bonn_person_tracking_compare"
mkdir -p "$SAVE_ROOT"
rm -rf "$SAVE_ROOT/droidw_baseline"
cp -a Outputs/Bonn/bonn_person_tracking "$SAVE_ROOT/droidw_baseline"
echo "Saved baseline to /content/vggt-demo/outputs_compare/droidw_baseline"
echo "Saved baseline backup to $SAVE_ROOT/droidw_baseline"


In [ ]:
from pathlib import Path
for name in ['droidw_ours', 'droidw_baseline']:
    root = Path('/content/vggt-demo/outputs_compare') / name
    print('
====', name, '====')
    print('exists:', root.exists())
    for p in [
        root / 'gt_poses.txt',
        root / 'traj/est_poses_full.txt',
        root / 'traj/metrics_full_traj.txt',
        root / 'traj/full_traj_2d.png',
        root / 'timer_summary.csv',
    ]:
        print(p, 'exists:', p.exists())
